# Goal 4 · Hybrid retrieval with Reciprocal Rank Fusion

**Goal:** restore Reciprocal Rank Fusion so both vector stores return ranked hits with a fused score.

**File:** `src/opensearch_client.py` (`hybrid_vector_search`)  |  **Evidence:** `retrieval_result.json`, `comparison_results.json`  |  **Needs goal 3 done first.**

### Concepts

- **The two retrievers.** BM25 (keyword) finds exact terms such as *Nexus Tower*. kNN (vector) finds meaning such as *catastrophic risk ≈ disaster exclusion*. Hybrid retrieval runs both.
- **Why not add the scores?** BM25 scores are unbounded (5, 12, …) and kNN cosine scores sit around 0–1. In a `bool/should`, the scales get mixed, and in this code the score is then dropped altogether (`_to_doc` keeps no `score`).
- **Reciprocal Rank Fusion** ignores raw scores and uses only **rank**: every list contributes `1 / (k + rank)` for each document, with `k = 60`. A document found by **both** retrievers collects two contributions and rises to the top.

In [ ]:
# Warm-up, no AWS needed: RRF by hand on two toy ranked lists
bm25 = ['A', 'B', 'C']          # keyword ranking
knn  = ['C', 'A', 'D']          # vector ranking
k = 60
fused = {}
for ranking in (bm25, knn):
    for rank, doc in enumerate(ranking, start=1):
        fused[doc] = fused.get(doc, 0.0) + 1.0 / (k + rank)
for doc, score in sorted(fused.items(), key=lambda x: x[1], reverse=True):
    print(doc, round(score, 5))
print('best possible score with two lists = 2/61 =', round(2 / 61, 5))

`A` (ranks 1 and 2) beats `C` (ranks 3 and 1) by a hair, and both beat documents only one retriever found. Now the real thing.

In [ ]:
import os, sys, traceback
sys.path.insert(0, os.path.abspath('.'))
import labkit
from src.graph_client import GraphLabDriver
from src.opensearch_client import OpenSearchVectorStore
from src.retriever import SemanticGraphRetriever

lab = GraphLabDriver()
os_store = OpenSearchVectorStore(); os_store.wait_until_ready()
retriever = SemanticGraphRetriever(lab, os_store)
Q = 'Nexus Tower commercial property exclusions'     # the evidence probe

try:
    print(retriever.hybrid_rag_traversal(Q, top_k=2))
except Exception:
    traceback.print_exc(limit=2)

### Observe: the error surfaces in `retriever.py`, but the cause is elsewhere

`hybrid_rag_traversal` reads `hit["score"]` from every result. The hits come from `OpenSearchVectorStore.hybrid_vector_search`, which never creates a score. Look at the two retrievers on their own: they already work.

In [ ]:
vec = lab.get_embedding(Q)
for name, hits in (('BM25', os_store._bm25(Q, 10)), ('kNN ', os_store._knn(vec, 10))):
    print(name, [h['chunk_id'] for h in hits])
labkit.show('src/opensearch_client.py', 'def hybrid_vector_search', after=14)

Seeing the same `chunk_id` more than once? **OpenSearch Serverless ignores client-supplied document ids**, so every re-seed (notebook 00, then goal 3) indexes another copy of a chunk. Fusion must count each chunk **once per list**.

## Task 4 · Rewrite `hybrid_vector_search`

Keep the signature `(self, query_text, query_vector, top_k: int = 3, rrf_k: int = 60)`. Requirements:

1. Run BM25 (`self._bm25`) and kNN (`self._knn`) **separately**, each with a candidate pool larger than `top_k` (use `max(top_k * 4, 10)`).
2. In each list, walk the hits in order. Skip a `chunk_id` already seen *in that list*, and give each new chunk the next rank (1, 2, 3 …).
3. Add `1.0 / (rrf_k + rank)` to that chunk's fused entry, keeping the document's fields (`chunk_id`, `text`, `source_uri`, `entity_ids`) plus a numeric `score`.
4. Return the `top_k` entries with the highest `score`, highest first.

<details><summary>Hint 1</summary>A dict keyed by <code>chunk_id</code> holds the fused entries: <code>fused.setdefault(cid, {**doc, "score": 0.0})</code>.</details>
<details><summary>Hint 2</summary>Rank counts <i>distinct</i> chunks in a list, so increment it only after the duplicate check.</details>
<details><summary>Hint 3</summary><code>sorted(fused.values(), key=lambda d: d["score"], reverse=True)[:top_k]</code></details>

In [ ]:
opensearch_client, retriever_mod = labkit.reload('opensearch_client', 'retriever')
os_store = opensearch_client.OpenSearchVectorStore(); os_store.wait_until_ready()
retriever = retriever_mod.SemanticGraphRetriever(lab, os_store)

hits = os_store.hybrid_vector_search(Q, vec, top_k=2)
for h in hits:
    print(h['chunk_id'], round(h['score'], 6))

# Recompute by hand from the raw lists and compare
expected = {}
for ranking in (os_store._bm25(Q, 10), os_store._knn(vec, 10)):
    seen = []
    for h in ranking:
        if h['chunk_id'] not in seen:
            seen.append(h['chunk_id'])
    for rank, cid in enumerate(seen, start=1):
        expected[cid] = expected.get(cid, 0.0) + 1 / (60 + rank)
print('hand-computed:', {c: round(s, 6) for c, s in sorted(expected.items(), key=lambda x: -x[1])})

for r in retriever.hybrid_rag_traversal(Q, top_k=2):
    print(f"\n{r['chunk_id']} score={r['score']:.4f}  grounded: {[e['entity_name'] + ' -> ' + str(e['applicable_metrics']) for e in r['grounded_entities']]}")

## Publish the evidence: retrieval probe and benchmark

`src.benchmark` runs three queries against **both** vector stores. It uses OpenSearch hybrid RRF and pgvector kNN, and records the document count, p50 latency and hits per query.

In [ ]:
!{sys.executable} -m src.emit_evidence retrieval
labkit.upload('retrieval_result.json')
!{sys.executable} -m src.benchmark
labkit.upload('comparison_results.json')
labkit.grade(4)

Now the full answer-key run works too. It re-seeds, then runs Test A (governed-metric resolution) and Test B (OpenSearch + graph traversal).

In [ ]:
!{sys.executable} -m src.main

### Debrief

- RRF needs **no score calibration**, which is why it's the default fusion method for hybrid search. `k = 60` damps the advantage of the top ranks, so agreement between retrievers matters more than being first in one of them.
- The benchmark compares an OpenSearch hybrid (BM25 + kNN) with pgvector's pure kNN over the same Titan vectors. With two chunks, latency dominates the comparison. Relevance differences only appear at scale.
- Recent OpenSearch versions also do fusion server-side (search pipelines with score normalisation or RRF). Client-side fusion keeps the logic visible and portable across both stores.